# CNN/DailyMail Data Preprocessing

1. Download and clean a reproducible CNN/DailyMail sample.
2. Build one fixed instruction and article prompt.
3. Build a structured target containing:
   - Summary
   - Article Facts: Who, What, Where, When, Result
   - Specialized Vocabulary Explanation
   - Exactly three quiz questions with answers
4. Keep the original records, but mark a pair as trainable only when the
   complete prompt and complete target fit within 2,048 tokens and the target
   is at most 450 tokens.

Long articles are not shortened here. Long test articles are handled by the
fine-tuning notebook with sentence-based chunking.


## 0. Setup

In [1]:
import os
import re
import json
import random
from collections import Counter

import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
RAW_PATH = os.path.join(RAW_DIR, "cnn_dailymail_raw.jsonl")
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "cnn_processed.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-1.3b"
SAMPLE_SIZE = 2500
SEED = 42

# OPT-1.3B uses a 2,048-token context window.
MAX_TRAIN_LENGTH = 2048
MAX_TARGET_TOKENS = 450

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
random.seed(SEED)

print(f"Raw output:       {RAW_PATH}")
print(f"Processed output: {OUTPUT_PATH}")


Raw output:       data/raw\cnn_dailymail_raw.jsonl
Processed output: data/processed\cnn_processed.jsonl


## 1. Download CNN/DailyMail

The dataset fields are:

- `article`: full news article
- `highlights`: reference summary/highlights
- `id`: article identifier

Streaming is used so the notebook does not download the full training set.

In [2]:
from datasets import load_dataset

if os.path.exists(RAW_PATH):
    print(f"Raw CNN/DailyMail file already exists at '{RAW_PATH}', skipping download.")
else:
    print("Streaming CNN/DailyMail 3.0.0 from Hugging Face...")
    dataset = load_dataset(
        "abisee/cnn_dailymail",
        "3.0.0",
        split="train",
        streaming=True,
    )

    dataset = dataset.shuffle(seed=SEED, buffer_size=10000)

    count = 0
    with open(RAW_PATH, "w", encoding="utf-8") as f:
        for row in dataset:
            record = {
                "article_id": row.get("id"),
                "article": row.get("article"),
                "summary": row.get("highlights"),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
            if count >= SAMPLE_SIZE:
                break

    print(f"Saved {count} raw CNN/DailyMail records to '{RAW_PATH}'")

Streaming CNN/DailyMail 3.0.0 from Hugging Face...
Saved 2500 raw CNN/DailyMail records to 'data/raw\cnn_dailymail_raw.jsonl'


## 2. Load, clean and remove duplicates

In [3]:
records = []
with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

raw_df = pd.DataFrame(records)
print(f"Loaded {len(raw_df)} raw records")


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    text = re.sub(r"[-*_=~]{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


clean_df = raw_df.dropna(subset=["article", "summary"]).copy()
clean_df["article"] = clean_df["article"].apply(clean_text)
clean_df["summary"] = clean_df["summary"].apply(clean_text)
clean_df = clean_df[
    clean_df["article"].astype(bool) &
    clean_df["summary"].astype(bool)
]
clean_df = clean_df.drop_duplicates(subset=["article"], keep="first")
clean_df = clean_df.reset_index(drop=True)

print(f"{len(clean_df)} records remain after cleaning")
clean_df.head()

Loaded 2500 raw records
2500 records remain after cleaning


,article_id,article,summary
0,9447bac3741cfc5c0ad64ba2716c6dc28018ae40,Celtic completed a deadline-busting swoop for ...,John Guidetti travelled to Glasgow for talks o...
1,8cd6e3574fc2e90608d5d977f2247675e23160b7,British taxpayers are donating thousands of po...,The Mtaani Inititative is part of a £6.7millio...
2,acbf542dc97fe262cabc8f82f852f2d14e80ac60,Manchester City have told UEFA chiefs that wea...,Premier League champions have met with UEFA pr...
3,1faf7efdcec81e90a5aa35fff2963e3bae7f605f,A picture has emerged of gun victim Reeva Stee...,Model looks every inch the accomplished shoote...
4,1cefec398cea732d452a1980dcc0744d9d6ee918,A 101-year-old woman has been evicted from her...,Texana Hollis's son failed to pay her property...


## 3. Build structured targets

CNN/DailyMail supplies an article and reference highlights. The additional
training sections are generated with simple, inspectable rules and use only
text from the article or its reference highlights.

Missing fact categories use `-`. Vocabulary terms must occur in the article.
Every target contains exactly three quiz question-and-answer pairs.


In [4]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
WORD_RE = re.compile(r"[A-Za-z][A-Za-z'-]+")

STOPWORDS = {
    "about", "after", "again", "against", "almost", "among", "because",
    "before", "being", "between", "could", "during", "every", "first",
    "from", "have", "into", "more", "most", "other", "over", "same",
    "should", "some", "such", "than", "that", "their", "there", "these",
    "they", "this", "those", "through", "under", "very", "what", "when",
    "where", "which", "while", "with", "would", "your"
}

MONTHS = (
    "January|February|March|April|May|June|July|August|September|"
    "October|November|December"
)
DAYS = "Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday"
WHEN_RE = re.compile(
    rf"\b(?:{MONTHS})(?:\s+\d{{1,2}}(?:st|nd|rd|th)?)?"
    rf"(?:,?\s+\d{{4}})?\b|\b(?:{DAYS})\b|\b(?:19|20)\d{{2}}\b|"
    r"\b(?:today|yesterday|tonight|last week|last month|this week)\b",
    re.IGNORECASE,
)
LOCATION_RE = re.compile(
    r"\b(?:in|at|near|from|outside|inside)\s+"
    r"((?:the\s+)?[A-Z][A-Za-z'-]*(?:\s+[A-Z][A-Za-z'-]*){0,3})"
)


def split_sentences(text: str):
    return [
        sentence.strip(" -•")
        for sentence in SENTENCE_SPLIT_RE.split(str(text).strip())
        if sentence.strip(" -•")
    ]


def normalize_point(text: str) -> str:
    text = re.sub(r"^[•\-*\d.)\s]+", "", str(text)).strip()
    if text and text[-1] not in ".!?":
        text += "."
    return text


def build_key_points(summary: str, article: str, maximum=5):
    points = []
    seen = set()

    for source in (summary, article):
        for sentence in split_sentences(source):
            point = normalize_point(sentence)
            key = point.lower()
            if len(point.split()) >= 5 and key not in seen:
                points.append(point)
                seen.add(key)
            if len(points) >= maximum:
                return points

    return points


def wordnet_definition(word: str):
    synsets = wn.synsets(word.lower())
    if not synsets:
        return None
    definition = synsets[0].definition().strip()
    return definition[0].upper() + definition[1:] if definition else None


def build_vocabulary(article: str, maximum=3):
    words = WORD_RE.findall(article)
    frequencies = Counter(word.lower() for word in words)
    candidates = []

    for word in sorted(frequencies, key=lambda w: (-len(w), frequencies[w], w)):
        if len(word) < 7 or word in STOPWORDS or word.endswith("'s"):
            continue

        definition = wordnet_definition(word)
        if definition:
            candidates.append({"term": word, "definition": definition})

        if len(candidates) >= maximum:
            break

    return candidates


def extract_subject(text: str):
    match = re.match(
        r"^([A-Z][A-Za-z'-]*(?:\s+[A-Z][A-Za-z'-]*){0,3})\b",
        str(text).strip(),
    )
    return match.group(1) if match else "-"


def extract_location(text: str):
    match = LOCATION_RE.search(str(text))
    return match.group(1).strip() if match else "-"


def extract_when(text: str):
    match = WHEN_RE.search(str(text))
    return match.group(0).strip() if match else "-"


def build_article_facts(summary: str, article: str):
    summary_sentences = split_sentences(summary)
    first = summary_sentences[0] if summary_sentences else "-"
    last = summary_sentences[-1] if len(summary_sentences) > 1 else "-"
    evidence = f"{summary} {article}"

    return {
        "who": extract_subject(first) if first != "-" else "-",
        "what": normalize_point(first) if first != "-" else "-",
        "where": extract_location(evidence),
        "when": extract_when(evidence),
        "result": normalize_point(last) if last != "-" else "-",
    }


def build_quiz(key_points, facts):
    candidates = [point for point in key_points if point and point != "-"]

    quiz = [
        {
            "question": "What is the main event reported in the article?",
            "answer": facts.get("what", "-"),
        },
        {
            "question": "What result or outcome is reported?",
            "answer": facts.get("result", "-"),
        },
    ]

    if facts.get("where", "-") != "-":
        third = {
            "question": "Where did the reported event happen?",
            "answer": facts["where"],
        }
    elif facts.get("when", "-") != "-":
        third = {
            "question": "When did the reported event happen?",
            "answer": facts["when"],
        }
    elif facts.get("who", "-") != "-":
        third = {
            "question": "Who is mainly involved in the article?",
            "answer": facts["who"],
        }
    else:
        third = {
            "question": "What other important fact is reported?",
            "answer": candidates[1] if len(candidates) > 1 else "-",
        }

    quiz.append(third)
    return quiz


def build_structured_target(row):
    key_points = build_key_points(row["summary"], row["article"])
    facts = build_article_facts(row["summary"], row["article"])
    vocabulary = build_vocabulary(row["article"])
    quiz = build_quiz(key_points, facts)

    vocabulary_text = "\n".join(
        f"{index}. {item['term']}: {item['definition']}"
        for index, item in enumerate(vocabulary, start=1)
    ) or "-"

    quiz_text = "\n".join(
        f"{index}. Question: {item['question']}\n"
        f"   Answer: {item['answer']}"
        for index, item in enumerate(quiz, start=1)
    )

    return (
        f"Summary:\n{row['summary']}\n\n"
        "Article Facts:\n"
        f"Who: {facts['who']}\n"
        f"What: {facts['what']}\n"
        f"Where: {facts['where']}\n"
        f"When: {facts['when']}\n"
        f"Result: {facts['result']}\n\n"
        "Specialized Vocabulary Explanation:\n"
        f"{vocabulary_text}\n\n"
        "Quiz:\n"
        f"{quiz_text}"
    )


structured_df = clean_df.copy()
structured_df["key_points"] = structured_df.apply(
    lambda row: build_key_points(row["summary"], row["article"]),
    axis=1,
)
structured_df["article_facts"] = structured_df.apply(
    lambda row: build_article_facts(row["summary"], row["article"]),
    axis=1,
)
structured_df["vocabulary"] = structured_df["article"].apply(build_vocabulary)
structured_df["quiz"] = structured_df.apply(
    lambda row: build_quiz(row["key_points"], row["article_facts"]),
    axis=1,
)
structured_df["training_target"] = structured_df.apply(
    build_structured_target,
    axis=1,
)

structured_df[
    ["summary", "article_facts", "vocabulary", "quiz", "training_target"]
].head(1)


,summary,article_facts,vocabulary,quiz,training_target
0,John Guidetti travelled to Glasgow for talks o...,"{'who': 'John Guidetti', 'what': 'John Guidett...","[{'term': 'disappointment', 'definition': 'A f...",[{'question': 'What is the main event reported...,Summary:\nJohn Guidetti travelled to Glasgow f...


## 4. Build the direct training prompt

In [5]:
TRAIN_PROMPT_TEMPLATE = """You are a news simplification assistant. Read the article and produce exactly the format below.

Summary:
<summary only>

Article Facts:
Who: <person or organization, or ->
What: <main event, or ->
Where: <location, or ->
When: <date or time, or ->
Result: <outcome, or ->

Specialized Vocabulary Explanation:
<up to three terms from the article with simple explanations, or ->

Quiz:
1. Question: <question>
   Answer: <answer>
2. Question: <question>
   Answer: <answer>
3. Question: <question>
   Answer: <answer>

Rules:
- Use only information available in the article.
- Keep the Summary section separate from all other sections.
- Use exactly the labels shown above.
- Write - when a fact category is not available.
- Include exactly three factual quiz questions with answers.

Article:
{article}

Response:
"""

CHUNK_PROMPT_TEMPLATE = """Read this article part and return only short factual notes.

Rules:
- Use only information in this article part.
- Preserve important names, actions, locations, dates, numbers and results.
- Use short bullet points.
- Do not write a summary, vocabulary section or quiz.

Article part:
{chunk}

Factual notes:
"""

NOTES_REDUCTION_PROMPT_TEMPLATE = """Shorten the factual notes below without dropping important names, actions, locations, dates, numbers or results.

Rules:
- Use only the supplied notes.
- Return only concise bullet-point factual notes.
- Do not add outside information.

Notes:
{notes}

Shorter factual notes:
"""

FINAL_PROMPT_TEMPLATE = """You are a news simplification assistant. The factual notes below come from consecutive parts of one article. Use only these notes and produce exactly the format below.

Summary:
<summary only>

Article Facts:
Who: <person or organization, or ->
What: <main event, or ->
Where: <location, or ->
When: <date or time, or ->
Result: <outcome, or ->

Specialized Vocabulary Explanation:
<up to three terms from the notes with simple explanations, or ->

Quiz:
1. Question: <question>
   Answer: <answer>
2. Question: <question>
   Answer: <answer>
3. Question: <question>
   Answer: <answer>

Rules:
- Use exactly the labels shown above.
- Keep the Summary section separate from all other sections.
- Write - when a fact category is not available.
- Include exactly three factual quiz questions with answers.

Factual notes:
{chunk_summaries}

Response:
"""

prompt_df = structured_df.copy()
prompt_df["prompt"] = prompt_df["article"].apply(
    lambda article: TRAIN_PROMPT_TEMPLATE.format(article=article)
)

print(prompt_df.loc[0, "prompt"][:1000])


You are a news simplification assistant. Read the article and produce exactly the format below.

Summary:
<summary only>

Article Facts:
Who: <person or organization, or ->
What: <main event, or ->
Where: <location, or ->
When: <date or time, or ->
Result: <outcome, or ->

Specialized Vocabulary Explanation:
<up to three terms from the article with simple explanations, or ->

Quiz:
1. Question: <question>
   Answer: <answer>
2. Question: <question>
   Answer: <answer>
3. Question: <question>
   Answer: <answer>

Rules:
- Use only information available in the article.
- Keep the Summary section separate from all other sections.
- Use exactly the labels shown above.
- Write - when a fact category is not available.
- Include exactly three factual quiz questions with answers.

Article:
Celtic completed a deadline-busting swoop for Stefan Scepovic but missed out on a second striker when a deal for Manchester City's John Guidetti collapsed just after midnight due to the paperwork failing to 

## 5. Check complete training-pair length

A record is trainable only when the complete target is at most 450 tokens and
the complete instruction, article and target pair is at most 2,048 tokens.
Tokenization uses `truncation=False`; no text is silently removed.


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def count_tokens(text: str, add_special_tokens: bool) -> int:
    return len(
        tokenizer(
            text,
            add_special_tokens=add_special_tokens,
            truncation=False,
        )["input_ids"]
    )


length_df = prompt_df.copy()
length_df["prompt_token_length"] = length_df["prompt"].apply(
    lambda text: count_tokens(text, add_special_tokens=True)
)
length_df["target_token_length"] = length_df["training_target"].apply(
    lambda text: count_tokens(
        text + tokenizer.eos_token,
        add_special_tokens=False,
    )
)
length_df["total_training_tokens"] = (
    length_df["prompt_token_length"] +
    length_df["target_token_length"]
)
length_df["fits_training_limit"] = (
    (length_df["target_token_length"] <= MAX_TARGET_TOKENS) &
    (length_df["total_training_tokens"] <= MAX_TRAIN_LENGTH)
)

fit_count = int(length_df["fits_training_limit"].sum())
print(
    f"Complete training pairs that fit: {fit_count}/{len(length_df)} "
    f"({fit_count / len(length_df):.2%})"
)
print(
    "Long/non-fitting records retained for possible test-time chunking:",
    int((~length_df["fits_training_limit"]).sum()),
)

length_df[
    [
        "prompt_token_length",
        "target_token_length",
        "total_training_tokens",
        "fits_training_limit",
    ]
].describe()

Complete training pairs that fit: 2284/2500 (91.36%)
Long/non-fitting records retained for possible test-time chunking: 216


,prompt_token_length,target_token_length,total_training_tokens
count,2500.000000,2500.000000,2500.0000
mean,1047.901600,281.449600,1329.3512
std,439.146096,41.477652,453.2746
min,319.000000,177.000000,526.0000
25%,724.000000,253.000000,993.0000
50%,948.000000,273.000000,1226.0000
75%,1266.250000,303.000000,1558.7500
max,2590.000000,555.000000,2946.0000


## 6. Save processed records and prompt templates

In [7]:
final_columns = [
    "article_id",
    "article",
    "summary",
    "prompt",
    "training_target",
    "key_points",
    "article_facts",
    "vocabulary",
    "quiz",
    "prompt_token_length",
    "target_token_length",
    "total_training_tokens",
    "fits_training_limit",
]

final_df = length_df[final_columns].reset_index(drop=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in final_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with open(TEMPLATES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "train_prompt_template": TRAIN_PROMPT_TEMPLATE,
            "single_prompt_template": TRAIN_PROMPT_TEMPLATE,
            "chunk_prompt_template": CHUNK_PROMPT_TEMPLATE,
            "notes_reduction_prompt_template": NOTES_REDUCTION_PROMPT_TEMPLATE,
            "final_prompt_template": FINAL_PROMPT_TEMPLATE,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(final_df)} records to '{OUTPUT_PATH}'")
print(f"Saved prompt templates to '{TEMPLATES_PATH}'")


Saved 2500 records to 'data/processed\cnn_processed.jsonl'
Saved prompt templates to 'data/processed\prompt_templates.json'
